# LeetCode 3441 — Minimum Cost Good Caption bằng Python

- Link: https://leetcode.com/problems/minimum-cost-good-caption/
- Độ khó: **Hard**
- Chủ đề: **Quy hoạch động hậu tố** với trạng thái `(vị trí, ký tự trước, độ dài khối hiện tại ∈ {1, 2, ≥3})`, **dựng lại tham lam** để được xâu **nhỏ nhất theo thứ tự từ điển** trong các xâu chi phí nhỏ nhất

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minCostGoodCaption(self, caption: str) -> str:
```

## 1. Mục tiêu học

1. Hiểu chi phí đổi ký tự `x → y` đúng bằng `|x − y|` (mỗi thao tác dịch một bước trong bảng chữ cái).
2. Mô hình "mọi ký tự xuất hiện thành khối ≥ 3 liên tiếp" bằng trạng thái **độ dài khối hiện tại** cắt ở 3.
3. Tính **DP hậu tố** `f[i][c][k]` = chi phí nhỏ nhất để điền `i..n−1` khi ký tự trước là `c` với độ dài khối `k`; chuyển sang ký tự khác dùng **min và min thứ hai** để đạt `O(26)` mỗi vị trí.
4. Dựng xâu **nhỏ nhất theo từ điển** bằng cách đi từ trái, mỗi bước chọn ký tự **nhỏ nhất** vẫn giữ được chi phí tối ưu (nhờ DP hậu tố).
5. Kiểm chứng bằng vét cạn và một DP theo khối độ dài 3–5.

## 2. Đề bài & ví dụ

Xâu **tốt** là xâu mà mọi ký tự xuất hiện thành các khối **ít nhất 3** ký tự liên tiếp. Mỗi thao tác đổi một ký tự thành ký tự liền trước hoặc liền sau trong bảng chữ cái. Biến `caption` thành xâu tốt với **ít thao tác nhất**; nếu nhiều xâu cùng chi phí, trả về xâu **nhỏ nhất theo thứ tự từ điển**; không thể thì `""`.

**Ví dụ 1:** `"cdcd"` → `"cccc"`. **Ví dụ 2:** `"aca"` → `"aaa"`. **Ví dụ 3:** `"bc"` → `""`.

**Ràng buộc:** `1 ≤ n ≤ 5·10^4`.

In [1]:
examples = [("cdcd", "cccc"), ("aca", "aaa"), ("bc", "")]

def is_good(t):
    i, n = 0, len(t)
    while i < n:
        j = i
        while j < n and t[j] == t[i]:
            j += 1
        if j - i < 3:
            return False
        i = j
    return True

def change_cost(s, t):
    return sum(abs(ord(a) - ord(b)) for a, b in zip(s, t))

for s, exp in examples:
    print(repr(s), '->', repr(exp), '| tốt?', is_good(exp) if exp else '-', '| chi phí', change_cost(s, exp) if exp else '-')

'cdcd' -> 'cccc' | tốt? True | chi phí 2
'aca' -> 'aaa' | tốt? True | chi phí 2
'bc' -> '' | tốt? - | chi phí -


## 3. Kiến thức nền

### 3.1. Chi phí đổi một ký tự

Đổi `x` thành `y` cần đúng `|x − y|` thao tác (không có cách nào ngắn hơn, và các ký tự độc lập nhau). Vậy chi phí biến `s` thành `t` là `Σ |s_i − t_i|`.

### 3.2. Phân tích xâu tốt thành khối

Xâu tốt = dãy các khối (run) độ dài ≥ 3. Khi điền từ trái sang, chỉ cần biết **ký tự cuối** và **độ dài khối hiện tại** (1, 2 hoặc "≥ 3") để quyết định ký tự tiếp theo có hợp lệ không:

- tiếp tục cùng ký tự: luôn được, độ dài tăng (cắt ở 3);
- đổi ký tự: chỉ được khi khối hiện tại đã ≥ 3.

In [2]:
def runs(t):
    out, i = [], 0
    while i < len(t):
        j = i
        while j < len(t) and t[j] == t[i]:
            j += 1
        out.append((t[i], j - i)); i = j
    return out

print(runs("aaabbbbcccc"), is_good("aaabbbbcccc"), runs("aabbb"), is_good("aabbb"))

[('a', 3), ('b', 4), ('c', 4)] True [('a', 2), ('b', 3)] False


### 3.3. Thứ tự từ điển + chi phí tối ưu: dựng tham lam với DP hậu tố

Nếu biết `f[i][trạng thái]` = chi phí nhỏ nhất cho **phần còn lại** từ vị trí `i`, ta dựng xâu từ trái: tại mỗi vị trí thử ký tự từ `'a'` lên, chọn ký tự **đầu tiên** mà `chi phí tại i + f[i + 1][trạng thái mới]` bằng tối ưu. Vì quyết định sớm hơn quan trọng hơn trong thứ tự từ điển, tham lam này cho xâu nhỏ nhất.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 `"cdcd"`: `n = 4` < 6 nên chỉ có **một** khối (không thể có hai khối ≥ 3). Một khối ký tự `y` có chi phí `|c−y| + |d−y| + |c−y| + |d−y|`: `y = c` hoặc `y = d` đều cho 2 ⇒ chọn `c` (nhỏ hơn) ⇒ `"cccc"`.

Ví dụ 2 `"aca"`: một khối `y`: `y = a` cho `0 + 2 + 0 = 2`, `y = b` cho `1 + 1 + 1 = 3` ⇒ `"aaa"`.

Ví dụ 3 `"bc"`: độ dài < 3 ⇒ không thể.

In [3]:
for y in 'bcde':
    print('cdcd ->', y * 4, 'chi phí', change_cost('cdcd', y * 4))

cdcd -> bbbb chi phí 6
cdcd -> cccc chi phí 2
cdcd -> dddd chi phí 2
cdcd -> eeee chi phí 6


## 5. Cách 1 — Brute Force: thử mọi xâu tốt trong khoảng `[min, max]` của các ký tự

Ký tự tối ưu luôn nằm trong `[min(s), max(s)]` (đưa một ký tự ra ngoài khoảng này rồi "kéo" về biên làm chi phí giảm mà vẫn giữ các khối). Thử mọi xâu, giữ `(chi phí, xâu)` nhỏ nhất.

In [4]:
import itertools

def good_brute(s):
    n = len(s)
    letters = [chr(c) for c in range(ord(min(s)), ord(max(s)) + 1)]
    best = None
    for tup in itertools.product(letters, repeat=n):
        t = ''.join(tup)
        if is_good(t):
            cand = (change_cost(s, t), t)
            if best is None or cand < best:
                best = cand
    return best[1] if best else ''

for s, exp in examples:
    print(repr(good_brute(s)), '| kỳ vọng', repr(exp))

'cccc' | kỳ vọng 'cccc'
'aaa' | kỳ vọng 'aaa'
'' | kỳ vọng ''


**Độ phức tạp:** `O(σ^n · n)` — chỉ dùng cho `n ≤ 7`.

## 6. Ý tưởng thuật toán / bước đột phá

Trạng thái sau khi điền xong vị trí `i − 1`: ký tự `c` và độ dài khối `k ∈ {1, 2, 3}` (3 nghĩa là ≥ 3). Định nghĩa

`f[i][c][k]` = chi phí nhỏ nhất điền `i..n−1` sao cho toàn xâu tốt.

- `f[n][c][k] = 0` nếu `k = 3`, ngược lại `∞`.
- `f[i][c][k] = min( |s_i − c| + f[i+1][c][min(k+1, 3)],  (nếu k = 3)  min_{d ≠ c} ( |s_i − d| + f[i+1][d][1] ) )`.

Phần `min_{d ≠ c}` tính chung cho mọi `c` bằng **giá trị nhỏ nhất và nhỏ thứ hai** của `g[d] = |s_i − d| + f[i+1][d][1]`.

Vị trí 0 không có ký tự trước: chọn `d` tối ưu với `|s_0 − d| + f[1][d][1]`. Dựng lại tham lam như mục 3.3. Tổng `O(26 · 3 · n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP theo khối độ dài 3–5 (chỉ tính **chi phí** nhỏ nhất)

Mọi xâu tốt có thể chia thành các khối **một ký tự** độ dài 3, 4 hoặc 5 (khối dài ≥ 6 tách được thành các khối 3–5; hai khối kề có thể cùng ký tự). `h[i]` = chi phí nhỏ nhất cho hậu tố `i`: `h[i] = min_{L ∈ {3,4,5}} ( min_y Σ_{j<L} |s_{i+j} − y| + h[i + L] )`. Cách này cho chi phí tối ưu (dùng để kiểm chứng), còn tie-break từ điển giữa các khối cùng ký tự thì rắc rối hơn — để cho cách 3.

In [5]:
def min_cost_blocks(s):
    n = len(s)
    a = [ord(ch) - 97 for ch in s]
    INF = float('inf')
    h = [INF] * (n + 1)
    h[n] = 0
    for i in range(n - 3, -1, -1):
        for L in (3, 4, 5):
            if i + L > n or h[i + L] == INF:
                continue
            seg = a[i:i + L]
            c = min(sum(abs(x - y) for x in seg) for y in range(min(seg), max(seg) + 1))
            h[i] = min(h[i], c + h[i + L])
    return h[0] if h[0] < INF else -1

for s, exp in examples:
    print(s, '-> chi phí nhỏ nhất', min_cost_blocks(s), '| chi phí của đáp án', change_cost(s, exp) if exp else -1)

cdcd -> chi phí nhỏ nhất 2 | chi phí của đáp án 2
aca -> chi phí nhỏ nhất 2 | chi phí của đáp án 2
bc -> chi phí nhỏ nhất -1 | chi phí của đáp án -1


### 7.2. Cách 3 — DP hậu tố theo trạng thái + dựng tham lam (lời giải chính)

In [6]:
INF = float('inf')

def good_dp(s):
    n = len(s)
    if n < 3:
        return ''
    a = [ord(ch) - 97 for ch in s]
    f = [None] * (n + 1)
    f[n] = [0 if k == 2 else INF for c in range(26) for k in range(3)]   # chỉ số c*3 + (k-1)
    for i in range(n - 1, 0, -1):
        nx, x = f[i + 1], a[i]
        g = [abs(x - d) + nx[d * 3] for d in range(26)]
        b1 = b2 = INF; i1 = -1
        for d in range(26):
            v = g[d]
            if v < b1:
                b2, b1, i1 = b1, v, d
            elif v < b2:
                b2 = v
        cur = [INF] * 78
        for c in range(26):
            cc = abs(x - c)
            cur[c * 3] = cc + nx[c * 3 + 1]
            cur[c * 3 + 1] = cc + nx[c * 3 + 2]
            v = cc + nx[c * 3 + 2]
            alt = b2 if i1 == c else b1
            cur[c * 3 + 2] = v if v < alt else alt
        f[i] = cur
    best, first = INF, -1
    for d in range(26):
        v = abs(a[0] - d) + f[1][d * 3]
        if v < best:
            best, first = v, d
    if best == INF:
        return ''
    out, c, k = [first], first, 0
    for i in range(1, n):
        target, x, nx = f[i][c * 3 + k], a[i], f[i + 1]
        for d in range(26):
            if d == c:
                k2 = min(k + 1, 2)
            elif k == 2:
                k2 = 0
            else:
                continue
            if abs(x - d) + nx[d * 3 + k2] == target:
                out.append(d); c, k = d, k2
                break
    return ''.join(chr(97 + d) for d in out)

for s, exp in examples:
    print(repr(good_dp(s)), '| kỳ vọng', repr(exp))

'cccc' | kỳ vọng 'cccc'
'aaa' | kỳ vọng 'aaa'
'' | kỳ vọng ''


(Trong code, `k` được lưu dưới dạng `0, 1, 2` tương ứng độ dài 1, 2, ≥ 3.)

### 7.3. Cách 4 — Cùng DP nhưng vector hóa theo 26 ký tự bằng `numpy`

In [7]:
import numpy as np

def good_numpy(s):
    n = len(s)
    if n < 3:
        return ''
    a = np.frombuffer(s.encode(), dtype=np.uint8).astype(np.int64) - 97
    BIG = 10 ** 15
    letters = np.arange(26, dtype=np.int64)
    f = np.empty((n + 1, 26, 3), dtype=np.int64)
    f[n] = BIG
    f[n, :, 2] = 0
    for i in range(n - 1, 0, -1):
        nx = f[i + 1]
        cc = np.abs(letters - a[i])
        g = cc + nx[:, 0]
        order = np.argsort(g, kind='stable')
        b1, b2, i1 = g[order[0]], g[order[1]], order[0]
        alt = np.full(26, b1); alt[i1] = b2
        f[i, :, 0] = cc + nx[:, 1]
        f[i, :, 1] = cc + nx[:, 2]
        f[i, :, 2] = np.minimum(cc + nx[:, 2], alt)
    np.minimum(f, BIG, out=f)
    first_cost = np.abs(letters - a[0]) + f[1, :, 0]
    first = int(np.argmin(first_cost))
    if first_cost[first] >= BIG:
        return ''
    out, c, k = [first], first, 0
    for i in range(1, n):
        target = f[i, c, k]
        for d in range(26):
            if d == c:
                k2 = min(k + 1, 2)
            elif k == 2:
                k2 = 0
            else:
                continue
            if abs(int(a[i]) - d) + f[i + 1, d, k2] == target:
                out.append(d); c, k = d, k2
                break
    return ''.join(chr(97 + d) for d in out)

for s, exp in examples:
    print(repr(good_numpy(s)), '| kỳ vọng', repr(exp))

'cccc' | kỳ vọng 'cccc'
'aaa' | kỳ vọng 'aaa'
'' | kỳ vọng ''


## 8. TRACE từng bước

Trace phần dựng lại của cách 3 trên `"cdcdzzy"`: tại mỗi vị trí in trạng thái, chi phí tối ưu còn lại và ký tự được chọn.

In [8]:
def trace(s):
    n = len(s)
    a = [ord(ch) - 97 for ch in s]
    f = [None] * (n + 1)
    f[n] = [0 if k == 2 else INF for c in range(26) for k in range(3)]
    for i in range(n - 1, 0, -1):
        nx, x = f[i + 1], a[i]
        g = [abs(x - d) + nx[d * 3] for d in range(26)]
        srt = sorted(range(26), key=lambda d: g[d])
        cur = [INF] * 78
        for c in range(26):
            cc = abs(x - c)
            cur[c * 3] = cc + nx[c * 3 + 1]
            cur[c * 3 + 1] = cc + nx[c * 3 + 2]
            alt = g[srt[1]] if srt[0] == c else g[srt[0]]
            cur[c * 3 + 2] = min(cc + nx[c * 3 + 2], alt)
        f[i] = cur
    costs = [abs(a[0] - d) + f[1][d * 3] for d in range(26)]
    first = min(range(26), key=lambda d: (costs[d], d))
    print(f"tổng chi phí tối ưu = {costs[first]}; vị trí 0 chọn {chr(97 + first)!r}")
    c, k, out = first, 0, [first]
    for i in range(1, n):
        target = f[i][c * 3 + k]
        for d in range(26):
            if d == c: k2 = min(k + 1, 2)
            elif k == 2: k2 = 0
            else: continue
            if abs(a[i] - d) + f[i + 1][d * 3 + k2] == target:
                print(f"vị trí {i} ({s[i]!r}): trạng thái ({chr(97 + c)!r}, khối {k + 1 if k < 2 else '>=3'}), "
                      f"còn tối ưu {target} -> chọn {chr(97 + d)!r}")
                out.append(d); c, k = d, k2
                break
    res = ''.join(chr(97 + d) for d in out)
    print('kết quả:', res, '| tốt?', is_good(res), '| chi phí', change_cost(s, res))

trace("cdcdzzy")

tổng chi phí tối ưu = 3; vị trí 0 chọn 'c'
vị trí 1 ('d'): trạng thái ('c', khối 1), còn tối ưu 3 -> chọn 'c'
vị trí 2 ('c'): trạng thái ('c', khối 2), còn tối ưu 2 -> chọn 'c'
vị trí 3 ('d'): trạng thái ('c', khối >=3), còn tối ưu 2 -> chọn 'c'
vị trí 4 ('z'): trạng thái ('c', khối >=3), còn tối ưu 1 -> chọn 'z'
vị trí 5 ('z'): trạng thái ('z', khối 1), còn tối ưu 1 -> chọn 'z'
vị trí 6 ('y'): trạng thái ('z', khối 2), còn tối ưu 1 -> chọn 'z'
kết quả: cccczzz | tốt? True | chi phí 3


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def minCostGoodCaption(self, caption: str) -> str:
        n = len(caption)
        if n < 3:
            return ''
        a = [ord(ch) - 97 for ch in caption]
        INF = float('inf')
        f = [None] * (n + 1)
        f[n] = [0 if k == 2 else INF for c in range(26) for k in range(3)]
        for i in range(n - 1, 0, -1):
            nx, x = f[i + 1], a[i]
            b1 = b2 = INF
            i1 = -1
            for d in range(26):
                v = abs(x - d) + nx[d * 3]
                if v < b1:
                    b2, b1, i1 = b1, v, d
                elif v < b2:
                    b2 = v
            cur = [INF] * 78
            for c in range(26):
                cc = abs(x - c)
                cur[c * 3] = cc + nx[c * 3 + 1]
                stay = cc + nx[c * 3 + 2]
                cur[c * 3 + 1] = stay
                alt = b2 if i1 == c else b1
                cur[c * 3 + 2] = stay if stay < alt else alt
            f[i] = cur
        best, first = INF, -1
        for d in range(26):
            v = abs(a[0] - d) + f[1][d * 3]
            if v < best:
                best, first = v, d
        if best == INF:
            return ''
        out = [first]
        c, k = first, 0
        for i in range(1, n):
            target, x, nx = f[i][c * 3 + k], a[i], f[i + 1]
            for d in range(26):
                if d == c:
                    k2 = k + 1 if k < 2 else 2
                elif k == 2:
                    k2 = 0
                else:
                    continue
                if abs(x - d) + nx[d * 3 + k2] == target:
                    out.append(d)
                    c, k = d, k2
                    break
        return ''.join(chr(97 + d) for d in out)

for s, exp in examples:
    got = Solution().minCostGoodCaption(s)
    print(repr(got), 'OK' if got == exp else 'SAI')

'cccc' OK
'aaa' OK
'' OK


## 10. Vì sao thuật toán đúng

1. **Trạng thái đủ:** điều kiện "tốt" chỉ phụ thuộc vào việc mỗi khối có ≥ 3 ký tự; khi điền tiếp, chỉ cần biết ký tự cuối và độ dài khối cuối (cắt ở 3). Chi phí cộng tính theo từng vị trí.
2. **Công thức `f`:** tại vị trí `i` hoặc giữ ký tự `c` (khối dài thêm), hoặc (chỉ khi khối đã ≥ 3) bắt đầu khối mới với `d ≠ c`. Khối cuối phải ≥ 3 ⇒ điều kiện biên `f[n]`.
3. **Min thứ hai:** `min_{d ≠ c} g[d]` bằng min toàn cục nếu nó không đạt tại `c`, ngược lại bằng min thứ hai (nếu có nhiều `d` cùng đạt min thì min thứ hai bằng min — vẫn đúng).
4. **Từ điển:** gọi `T*` là xâu tối ưu nhỏ nhất theo từ điển. Tại vị trí `i`, mọi xâu tối ưu có tiền tố `T*[0..i)` có ký tự tại `i` thuộc tập "đạt tối ưu"; tham lam chọn ký tự nhỏ nhất trong tập này — đúng là `T*[i]` (nếu chọn lớn hơn thì xâu lớn hơn; nhỏ hơn thì không còn tối ưu). Quy nạp theo `i`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi xâu | `O(σ^n · n)` | `O(n)` |
| 2. Khối 3–5 (chi phí) | `O(n · 5 · 26)` | `O(n)` |
| 3. DP trạng thái + dựng | `O(78 n + 26 n)` | `O(78 n)` |
| 4. `numpy` | `O(n)` vòng lặp vector 26 phần tử | `O(78 n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'dp': good_dp, 'numpy': good_numpy, 'submit': lambda s: Solution().minCostGoodCaption(s)}
edge = [("a", ""), ("aaa", "aaa"), ("abc", "bbb"), ("aab", "aaa"), ("zzzz", "zzzz"),
        ("abcdef", "bbbeee"), ("aaabbb", "aaabbb"), ("aabbaa", "aaaaaa")]
for s, exp in examples + edge:
    assert good_brute(s) == exp, (s, good_brute(s))
    for name, f in impls.items():
        assert f(s) == exp, (name, s, f(s))
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn (và với DP khối về chi phí)

In [11]:
import random
rng = random.Random(3441)
cnt = 0
for _ in range(600):
    n = rng.randint(1, 7)
    s = ''.join(rng.choice('abcd') for _ in range(n))
    exp = good_brute(s)
    for name, f in impls.items():
        assert f(s) == exp, (name, s)
    cnt += 1
for _ in range(300):
    s = ''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(rng.randint(3, 60)))
    r = Solution().minCostGoodCaption(s)
    assert is_good(r) and change_cost(s, r) == min_cost_blocks(s) and r == good_numpy(s)
print('Khớp vét cạn trên', cnt, 'xâu ngắn; chi phí khớp DP khối trên 300 xâu dài hơn.')

Khớp vét cạn trên 600 xâu ngắn; chi phí khớp DP khối trên 300 xâu dài hơn.


### 12.3. Ca lớn `n = 5·10^4`

In [12]:
import time
n = 5 * 10 ** 4
big = {
    'ngẫu nhiên': ''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(n)),
    'xen kẽ az': 'az' * (n // 2),
    'toàn a': 'a' * n,
}
for name, s in big.items():
    t = time.time(); r = Solution().minCostGoodCaption(s); t1 = time.time() - t
    t = time.time(); r2 = good_numpy(s); t2 = time.time() - t
    assert r == r2 and is_good(r)
    print(f"{name:12s}: chi phí {change_cost(s, r):7d} | thuần Python {t1:.2f}s | numpy {t2:.2f}s")

ngẫu nhiên  : chi phí  183419 | thuần Python 0.45s | numpy 0.38s


xen kẽ az   : chi phí  416675 | thuần Python 0.42s | numpy 0.37s


toàn a      : chi phí       0 | thuần Python 0.26s | numpy 0.32s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Khi hòa chi phí, chọn ký tự **lớn** hơn (hoặc ký tự gặp đầu tiên theo thứ tự khác)

In [13]:
def wrong_tiebreak(s):
    # chọn trong các xâu chi phí nhỏ nhất xâu LỚN nhất (đảo thứ tự duyệt ký tự)
    n = len(s)
    letters = [chr(c) for c in range(ord(min(s)), ord(max(s)) + 1)]
    best = None
    for tup in itertools.product(letters, repeat=n):
        t = ''.join(tup)
        if is_good(t):
            cand = (change_cost(s, t), [-ord(ch) for ch in t])
            if best is None or cand < best:
                best = cand
    return ''.join(chr(-v) for v in best[1]) if best else ''

print('sai :', wrong_tiebreak("cdcd"), '| đúng:', Solution().minCostGoodCaption("cdcd"))

sai : dddd | đúng: cccc


**Vì sao sai:** đề yêu cầu xâu **nhỏ nhất** theo từ điển trong các xâu chi phí tối ưu.

### Lỗi 2 — Quên điều kiện khối cuối cùng phải dài ≥ 3

In [14]:
def wrong_last_block(s):
    n = len(s)
    a = [ord(ch) - 97 for ch in s]
    f = [None] * (n + 1)
    f[n] = [0] * 78                                   # SAI: mọi trạng thái kết thúc đều hợp lệ
    for i in range(n - 1, 0, -1):
        nx, x = f[i + 1], a[i]
        g = [abs(x - d) + nx[d * 3] for d in range(26)]
        cur = [INF] * 78
        for c in range(26):
            cc = abs(x - c)
            cur[c * 3] = cc + nx[c * 3 + 1]
            cur[c * 3 + 1] = cc + nx[c * 3 + 2]
            cur[c * 3 + 2] = min(cc + nx[c * 3 + 2], min(g[d] for d in range(26) if d != c))
        f[i] = cur
    first = min(range(26), key=lambda d: (abs(a[0] - d) + f[1][d * 3], d))
    out, c, k = [first], first, 0
    for i in range(1, n):
        target, nx = f[i][c * 3 + k], f[i + 1]
        for d in range(26):
            if d == c: k2 = min(k + 1, 2)
            elif k == 2: k2 = 0
            else: continue
            if abs(a[i] - d) + nx[d * 3 + k2] == target:
                out.append(d); c, k = d, k2; break
    return ''.join(chr(97 + d) for d in out)

r = wrong_last_block("aaabz")
print('sai :', r, '-> tốt?', is_good(r), '| đúng:', Solution().minCostGoodCaption("aaabz"))

sai : aaaaz -> tốt? False | đúng: aaaaa


**Vì sao sai:** khối **cuối** cũng phải có ≥ 3 ký tự; thiếu điều kiện biên `f[n][c][k] = ∞` với `k < 3` cho ra xâu không tốt.

### Lỗi 3 — Biến cả xâu thành **một** ký tự (trung vị)

In [15]:
def wrong_single_char(s):
    if len(s) < 3:
        return ''
    best = min((change_cost(s, ch * len(s)), ch) for ch in 'abcdefghijklmnopqrstuvwxyz')
    return best[1] * len(s)

s = "aaazzz"
print('sai :', wrong_single_char(s), 'chi phí', change_cost(s, wrong_single_char(s)),
      '| đúng:', Solution().minCostGoodCaption(s), 'chi phí', change_cost(s, Solution().minCostGoodCaption(s)))

sai : aaaaaa chi phí 75 | đúng: aaazzz chi phí 0


**Vì sao sai:** xâu tốt có thể gồm nhiều khối khác nhau; ép một ký tự cho cả xâu thường tốn hơn nhiều.

### Lỗi 4 — Dựng lại từ **phải sang trái** (DP tiền tố) rồi chọn ký tự nhỏ nhất ở cuối trước

In [16]:
def wrong_backward(s):
    # DP tiền tố: p[i][c][k] = chi phí nhỏ nhất cho s[0..i] kết thúc bằng c với độ dài khối k.
    # Dựng ngược: ở mỗi bước chọn ký tự NHỎ NHẤT còn tối ưu — tối ưu cho hậu tố, không cho tiền tố.
    n = len(s)
    if n < 3:
        return ''
    a = [ord(ch) - 97 for ch in s]
    p = [[[INF] * 3 for _ in range(26)] for _ in range(n)]
    for c in range(26):
        p[0][c][0] = abs(a[0] - c)
    for i in range(1, n):
        best3 = [(p[i - 1][d][2], d) for d in range(26)]
        srt = sorted(best3)
        for c in range(26):
            cc = abs(a[i] - c)
            alt = srt[1][0] if srt[0][1] == c else srt[0][0]
            p[i][c][0] = cc + alt
            p[i][c][1] = cc + p[i - 1][c][0]
            p[i][c][2] = cc + min(p[i - 1][c][1], p[i - 1][c][2])
    tot = min(p[n - 1][c][2] for c in range(26))
    if tot == INF:
        return ''
    c = min(d for d in range(26) if p[n - 1][d][2] == tot)
    k, out = 2, [c]
    for i in range(n - 1, 0, -1):
        need = p[i][c][k] - abs(a[i] - c)
        if k == 0:
            c = min(d for d in range(26) if d != c and p[i - 1][d][2] == need); k = 2
        elif k == 1:
            k = 0
        else:
            k = 1 if p[i - 1][c][1] == need else 2
        out.append(c)
    return ''.join(chr(97 + d) for d in reversed(out))

rng = random.Random(4)
ce = None
for _ in range(20000):
    s = ''.join(rng.choice('abcd') for _ in range(rng.randint(3, 7)))
    if wrong_backward(s) != good_brute(s):
        ce = s; break
print('phản ví dụ:', ce)
print('sai :', wrong_backward(ce), '| đúng:', Solution().minCostGoodCaption(ce), '| cùng chi phí?',
      change_cost(ce, wrong_backward(ce)) == change_cost(ce, Solution().minCostGoodCaption(ce)))

phản ví dụ: cabccba
sai : ccccbbb | đúng: bbbbbbb | cùng chi phí? True


**Vì sao sai:** chọn ký tự nhỏ nhất ở **cuối** xâu trước không đảm bảo xâu nhỏ nhất theo từ điển — thứ tự từ điển ưu tiên vị trí **đầu**. Phải tính DP **hậu tố** và dựng từ **trái sang**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chi phí nhỏ nhất, hòa thì xâu nhỏ nhất theo từ điển" ⇒ DP **hậu tố** + dựng tham lam từ trái.
- Ràng buộc "khối ≥ K" ⇒ trạng thái độ dài khối cắt ở `K`.
- Chuyển sang "ký tự bất kỳ khác `c`" ⇒ min và min thứ hai.
- Cận chi phí theo khối độ dài `K..2K−1` là một cách kiểm chứng độc lập.

## 15. Biến thể của bài

1. Khối ≥ `K` bất kỳ ⇒ trạng thái độ dài `1..K`.
2. Chi phí đổi ký tự theo bảng `w[x][y]` bất kỳ ⇒ thay `|x − y|` bằng `w`.
3. Chỉ cần **chi phí** (không cần xâu) ⇒ DP khối 3–5 là đủ.
4. Bài liên quan: LC 1531 (String Compression II), LC 2167 — DP trên khối.

In [17]:
# Biến thể 1: khối >= K, chỉ tính chi phí, đối chiếu vét cạn
def min_cost_blocks_K(s, K):
    n = len(s); a = [ord(ch) - 97 for ch in s]
    h = [INF] * (n + 1); h[n] = 0
    for i in range(n - K, -1, -1):
        for L in range(K, 2 * K):
            if i + L > n or h[i + L] == INF:
                continue
            seg = a[i:i + L]
            c = min(sum(abs(x - y) for x in seg) for y in range(min(seg), max(seg) + 1))
            h[i] = min(h[i], c + h[i + L])
    return h[0] if h[0] < INF else -1

def brute_K(s, K):
    letters = [chr(c) for c in range(ord(min(s)), ord(max(s)) + 1)]
    best = -1
    for tup in itertools.product(letters, repeat=len(s)):
        t = ''.join(tup)
        if all(L >= K for _, L in runs(t)):
            c = change_cost(s, t)
            best = c if best == -1 else min(best, c)
    return best

rng = random.Random(15)
for _ in range(300):
    K = rng.randint(1, 4)
    s = ''.join(rng.choice('abcd') for _ in range(rng.randint(1, 7)))
    assert min_cost_blocks_K(s, K) == brute_K(s, K), (s, K)
print('Biến thể khối >= K (chi phí) khớp vét cạn.')

Biến thể khối >= K (chi phí) khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi xâu | mũ | Oracle |
| 2 | Khối 3–5 | `O(n · 26 · 5)` | Chỉ chi phí |
| 3 | DP trạng thái + dựng tham lam | `O(104 n)` | **Nộp bài** |
| 4 | Như 3, `numpy` | `O(n)` vector | Thay thế |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `"abcabc"`.

**Bài 2 (tính tay):** `"zzab"`.

**Bài 3 (điền TODO):** viết `min_cost_only(s)` — chỉ trả về chi phí (hoặc `-1`).

**Bài 4 (tự viết lại):** biến thể 1 với `K` bất kỳ, trả về xâu nhỏ nhất theo từ điển.

**Bài 5 (mini challenge):** biến thể 2 — chi phí đổi `x → y` là `w[x][y]` cho trước (ma trận 26×26).

In [18]:
def min_cost_only_todo(s):
    # TODO
    raise NotImplementedError

def good_K_todo(s, K):
    # TODO
    raise NotImplementedError

def good_weighted_todo(s, w):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Hai khối độ dài 3 hoặc một khối 6. (2) Khối `"abc"` → `"bbb"` tốn 2. (3) `"bbbbbb"` tốn 4.

**Bài 2.** (1) Độ dài 4 ⇒ một khối. (2) Chi phí theo ký tự `y`: tổng khoảng cách. (3) Hòa ⇒ ký tự nhỏ hơn.

**Bài 3.** (1) Dùng cách 2. (2) Hoặc `min` của bước chọn ký tự đầu trong cách 3. (3) `n < 3` ⇒ `-1`.

**Bài 4.** (1) Trạng thái `k ∈ 1..K`. (2) Đổi ký tự khi `k = K`. (3) Biên `f[n][c][K] = 0`.

**Bài 5.** (1) Thay `abs(x − d)` bằng `w[x][d]`. (2) Phần min thứ hai giữ nguyên. (3) Kiểm tra bằng vét cạn với `w` ngẫu nhiên.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
a1 = Solution().minCostGoodCaption("abcabc")
a2 = Solution().minCostGoodCaption("zzab")
assert a1 == good_brute("abcabc") and a2 == good_brute("zzab")
print('Bài 1:', a1, '| Bài 2:', a2)

def min_cost_only(s):
    return min_cost_blocks(s)

for s in ["cdcd", "aca", "bc", "abcabc"]:
    r = Solution().minCostGoodCaption(s)
    assert min_cost_only(s) == (change_cost(s, r) if r else -1)
print('Bài 3:', [min_cost_only(s) for s in ["cdcd", "aca", "bc", "abcabc"]])

Bài 1: bbbbbb | Bài 2: bbbb
Bài 3: [2, 2, -1, 4]


In [20]:
# Bài 4, 5 — DP tổng quát: khối >= K, chi phí theo hàm w(x, d)
def good_general(s, K, w):
    n = len(s)
    if n < K:
        return ''
    a = [ord(ch) - 97 for ch in s]
    f = [None] * (n + 1)
    f[n] = [[0 if k == K - 1 else INF for k in range(K)] for c in range(26)]
    for i in range(n - 1, 0, -1):
        nx, x = f[i + 1], a[i]
        g = [w(x, d) + nx[d][0] for d in range(26)]
        srt = sorted(range(26), key=lambda d: g[d])
        cur = [[INF] * K for _ in range(26)]
        for c in range(26):
            cc = w(x, c)
            for k in range(K):
                v = cc + nx[c][min(k + 1, K - 1)]
                if k == K - 1:
                    alt = g[srt[1]] if srt[0] == c else g[srt[0]]
                    v = min(v, alt)
                cur[c][k] = v
        f[i] = cur
    costs = [w(a[0], d) + f[1][d][0] if n > 1 else (w(a[0], d) if K == 1 else INF) for d in range(26)]
    first = min(range(26), key=lambda d: (costs[d], d))
    if costs[first] == INF:
        return ''
    out, c, k = [first], first, 0
    for i in range(1, n):
        target, nx = f[i][c][k], f[i + 1]
        for d in range(26):
            if d == c: k2 = min(k + 1, K - 1)
            elif k == K - 1: k2 = 0
            else: continue
            if w(a[i], d) + nx[d][k2] == target:
                out.append(d); c, k = d, k2; break
    return ''.join(chr(97 + d) for d in out)

def brute_general(s, K, w):
    best = None
    for tup in itertools.product('abcd', repeat=len(s)):
        t = ''.join(tup)
        if all(L >= K for _, L in runs(t)):
            cand = (sum(w(ord(x) - 97, ord(y) - 97) for x, y in zip(s, t)), t)
            best = cand if best is None or cand < best else best
    return best[1] if best else ''

rng = random.Random(19)
W = [[rng.randint(0, 5) if x != y else 0 for y in range(26)] for x in range(26)]
for x in range(26):
    for y in range(4, 26):
        W[x][y] = 100                             # buộc lời giải chỉ dùng a..d để vét cạn được
for _ in range(300):
    K = rng.randint(2, 3)
    s = ''.join(rng.choice('abcd') for _ in range(rng.randint(1, 6)))
    assert good_general(s, K, lambda x, d: abs(x - d)) == brute_general(s, K, lambda x, d: abs(x - d)) if max(s) <= 'd' else True
    assert good_general(s, K, lambda x, d: W[x][d]) == brute_general(s, K, lambda x, d: W[x][d])
assert all(good_general(s, 3, lambda x, d: abs(x - d)) == Solution().minCostGoodCaption(s) for s in ["cdcd", "aca", "bc", "abcdef"])
print('Bài 4, 5 khớp vét cạn; K = 2 với "abab":', good_general("abab", 2, lambda x, d: abs(x - d)))

Bài 4, 5 khớp vét cạn; K = 2 với "abab": aaaa


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
f[n][c][k] = 0 nếu k = "≥3" else INF
f[i][c][k] = min(|s_i - c| + f[i+1][c][min(k+1, 3)],
                 (k = "≥3") ? min_{d≠c} |s_i - d| + f[i+1][d][1] : INF)     # min & min thứ hai
vị trí 0: d tốt nhất theo |s_0 - d| + f[1][d][1] (hòa -> d nhỏ)
dựng: từ trái, mỗi bước thử d = a..z, chọn d đầu tiên giữ đúng chi phí tối ưu
```

**Checklist:**

- [ ] `n < 3` ⇒ `""`?
- [ ] Khối cuối ≥ 3 (điều kiện biên)?
- [ ] Đổi ký tự chỉ khi khối hiện tại ≥ 3?
- [ ] DP **hậu tố** và dựng từ **trái** để được xâu nhỏ nhất?
- [ ] Hòa chi phí ⇒ chọn ký tự nhỏ hơn?